## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [9]:
import mlflow
from mlflow.tracking import MlflowClient

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

client = MlflowClient()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = client.get_experiment_by_name(settings.mlflow_experiment_name)

In [10]:
# Get all traces from last 7 days (adjust as needed)
filter_string = """
tags.graph_type = 'simulation' AND
tags.agent_name = 'action-responder'
"""
traces = client.search_traces(
    locations=[experiment.experiment_id],
    filter_string=filter_string,
    max_results=500,
)

In [ ]:
spans = [
    span
    for trace in traces
    for span in client.get_trace(trace.info.trace_id).data.spans
    if span.name == "action_response_agent"
]

records = [
    {"inputs": span.inputs["input_data"], "outputs": span.outputs} for span in spans
]

# dataset = client.create_dataset(
#     name="action_response_agent",
#     experiment_id=[experiment.experiment_id],
# )
# dataset.merge_records(records)

<EvaluationDataset: experiment_ids=['7'], profile=None, records=[DatasetRecord(dataset_id='d-6b83e4daad104e7390dd878e08839d59',
               inputs={'hidden_state': 'Patient is A&O x1. HR 112, RR 20, BP '
                                       '118/76. SCTM: Pale, cool, moist. '
                                       'Obvious deformity of the left '
                                       'clavicle, with superficial bleeding. '
                                       'Tenderness and swelling over the '
                                       'pelvis. No visible head trauma or '
                                       'airway compromise. SAMPLE: Allergies: '
                                       'None. Medications: None. Past trauma: '
                                       'None. Last intake: Water 30 minutes '
                                       'ago. Events: Fell while scrambling on '
                                       'loose rocks.',
                       'hidden_truth': 'Pat

[Trace(trace_id=tr-affe256ef8a85223f22a7b2f6f8953f9), Trace(trace_id=tr-02676ba69b9f82ffdc08ac94d18f06e2), Trace(trace_id=tr-141e739e58049c1a79786b59ee8fdd03), Trace(trace_id=tr-5edc75e07690dd965b51d8b35fb26a86), Trace(trace_id=tr-2ea58049cc996b1cccb1815df46cec62), Trace(trace_id=tr-fa63eaabbc1080b664d1dc4aef22e720), Trace(trace_id=tr-6979387d5e89563634062b84bb2f547c), Trace(trace_id=tr-35573d4c1b256e7ca341b17d06b2f062), Trace(trace_id=tr-79f172b56f8eddf75effd7bc01b520e1), Trace(trace_id=tr-050dbc90371a14aa2798144552bef0ac)]

In [12]:
from mlflow.genai.judges import make_judge

JUDGE_MODEL_ENDPOINT = "openrouter:/deepseek/deepseek-v3.2"

COMPLETION_JUDGE_INSTRUCTIONS = """\
You are evaluating the scoring accuracy and pedagogical quality of an
AI-generated response in a wilderness first responder (WFR) training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does completion_score align with the WFR rubric based on cumulative actions?
2. Is the score increase from previous turn reasonable (<=0.2 unless explicit bundling)?
3. Does the score always build on itself? Does completion_score never decrease across turns?
4. If there was no action from the student, the completion_score should not increase from it's previous turn.
    If it did without action, automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


completion_judge = make_judge(
    name="completion-judge",
    instructions=COMPLETION_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [13]:
QUALITY_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


Evaluate against these criteria:
1. Does narrative_text end with an open question? Does feedback contain NO questions?
2. Is the feedback personalized to the students response and focus only on history, not future?
3. Is the feedback encouraging and constructive without harsh corrections and focused
    on feedback alone?
4. Is feedback and narrative length between 2-4 sentences?
5. Does the narrative not overtly share hidden information, only revealing
    if the student's action constitutes it? If any extra information was injected in the narrative
    or feedback, automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


quality_judge = make_judge(
    name="quality-judge",
    instructions=QUALITY_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [14]:
MEDICAL_JUDGE_INSTRUCTIONS = """\
You are evaluating the medical accuracy of an AI-generated response in a \
wilderness first responder training simulation.

AI Input:
{{ inputs }}

DO NOT CONSIDER PREVIOUS INPUTS
Only evaluate the final AI agent output below:
{{ outputs }}


TIP: Reference hidden_truth and hidden_state to determine if treatment was premature.

Evaluate against these criteria:
1. Is was_correct accurate?
 - was_correct should be FALSE if student performed treatment \
    (splint, bandage, medication, move patient) without assessment
 - was_correct should be TRUE for assessment actions (vitals, exam, SAMPLE history)
2. Does was_correct accurately gage the quality of the student's action?
3. Is there anything in the AI response that is not medically accurate? If so,
    automatically provide a zero score for the agent (THIS SUPERSEDES ANY PREVIOUS ASSESSMENT).

Provide a score of 0-1 on how well the AI Agent performs
"""


medical_judge = make_judge(
    name="medical-judge",
    instructions=MEDICAL_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [ ]:
scorers = [completion_judge, quality_judge, medical_judge]
dataset = client.get_dataset(dataset_id="d-6b83e4daad104e7390dd878e08839d59")

# results = mlflow.genai.evaluate(
#     data=dataset,
#     scorers=scorers,
# )

In [16]:
from summit_sim.agents.action_responder import ActionRequest, action_response_agent


async def optimize_wrapper(**kwargs) -> dict:  # noqa: ANN003
    """Parse dict into pydantic request to pass to agent."""
    # 1. Pack MLflow's individual kwargs into your Pydantic model
    request_data = ActionRequest(**kwargs)

    # 2. Await your traced LangGraph agent
    response = await action_response_agent(input_data=request_data)

    # 3. Dump the Pydantic response to a dictionary for MLflow compatibility
    return response.model_dump()

In [ ]:
import os

from mlflow.genai.optimize.optimizers import MetaPromptOptimizer

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "True"

result = mlflow.genai.optimize_prompts(
    predict_fn=optimize_wrapper,
    train_data=dataset,
    prompt_uris=[
        "prompts:/action-responder-system@latest",
        "prompts:/action-responder-user@latest",
    ],
    optimizer=MetaPromptOptimizer(
        reflection_model=JUDGE_MODEL_ENDPOINT,
        guidelines="This prompt is for a Wilderness First Responder agent which dynamically responds to students actions in an emergency situation",
    ),
    scorers=scorers,
)

/home/bhamm/repos/summit-sim/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/03/30 06:53:08 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: 24 training examples provided, using few-shot metaprompting
2026/03/30 06:53:08 INFO mlflow.genai.optimize.optimizers.metaprompt_optimizer: Evaluating baseline prompts on training data...
2026-03-30 06:53:08 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=45
2026-03-30 06:53:08 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=3
2026-03-30 06:53:08 [INFO] summit_sim.agents.action_responder: Processing student action: turn=1/5, action_length=3
2026-03-30 06:53:08 [INFO] s

🏃 View run abrasive-skunk-177 at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/724313890bc341e0b319dad3d62b7850
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


Trace(trace_id=tr-9e78238b9abce9abf91438f79f9e6b6d)

In [19]:
optimized_system_prompt = result.optimized_prompts[0]
optimized_user_prompt = result.optimized_prompts[1]

print(f"Optimized system prompt URI: {optimized_system_prompt.uri}")
print(f"Optimized system template: {optimized_system_prompt.template}")
print(f"Optimized user prompt URI: {optimized_user_prompt.uri}")
print(f"Optimized user template: {optimized_user_prompt.template}")

Optimized system prompt URI: prompts:/action-responder-system/11
Optimized system template: You are an expert Wilderness First Responder (WFR) instructor guiding students through realistic wilderness emergency scenarios. Your goal is to help students learn proper assessment and treatment protocols while maintaining an engaging, supportive learning environment.

=== YOUR CORE TASK ===
For each student action:
1.  Determine `was_correct`: TRUE if the action is an assessment step (even if vague or incomplete) or FALSE if it is a treatment action without prior assessment (e.g., splinting, moving, giving medication). 'idk' or similar non-actions are considered assessment intent and should be TRUE to maintain encouragement.
2.  Calculate `completion_score` CUMULATIVELY based on ALL previous turns using the PAS rubric. The score must NEVER decrease. Only increase the score if the student's current action represents a substantive assessment step. If the student's action is 'idk' or a generic q

In [ ]:
models_to_test = [
    "google/gemini-3.1-flash-lite-preview",
    "qwen/qwen3.5-flash-02-23",
    "qwen/qwen3-235b-a22b-2507",
    "openai/gpt-4o-mini",
    "openai/gpt-4.1-mini",
    "openai/gpt-5-nano",
    "openai/gpt-5.4-nano",
    "openai/gpt-oss-120b",
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
    "xiaomi/mimo-v2-flash",
    "z-ai/glm-4.7-flash",
    "",
]